In [1]:
import random
import math
import time

def funcion_objetivo(x, y):
    return (x**2) * (4 - 2.1 * x**2 + (x**4) / 3.0) + x * y + (y**2) * (-4 + 4 * y**2)

def busqueda_estructurada(x_min=-2.0, x_max=2.0, y_min=-1.0, y_max=1.0, paso=0.001):
    mejor_x = x_min
    mejor_y = y_min
    mejor_valor = funcion_objetivo(mejor_x, mejor_y)

    evaluaciones = 0

    x = x_min
    while x <= x_max:
        y = y_min
        while y <= y_max:
            valor = funcion_objetivo(x, y)
            evaluaciones += 1

            if valor < mejor_valor:
                mejor_valor = valor
                mejor_x = x
                mejor_y = y

            y += paso
        x += paso

    return mejor_x, mejor_y, mejor_valor, evaluaciones

def crear_particula(x_min, x_max, y_min, y_max, ajuste_v=100.0):
    x = random.uniform(x_min, x_max)
    y = random.uniform(y_min, y_max)
    vx = random.uniform(x_min / ajuste_v, x_max / ajuste_v)
    vy = random.uniform(y_min / ajuste_v, y_max / ajuste_v)

    valor = funcion_objetivo(x, y)

    return {
        "x": x,
        "y": y,
        "vx": vx,
        "vy": vy,
        "mejor_x": x,
        "mejor_y": y,
        "mejor_valor": valor
    }

def actualizar_velocidad(particula, gbest_x, gbest_y, inercia, c1, c2):
    r1 = random.random()
    r2 = random.random()

    componente_cognitiva_x = c1 * r1 * (particula["mejor_x"] - particula["x"])
    componente_social_x    = c2 * r2 * (gbest_x - particula["x"])
    particula["vx"] = inercia * particula["vx"] + componente_cognitiva_x + componente_social_x

    r3 = random.random()
    r4 = random.random()

    componente_cognitiva_y = c1 * r3 * (particula["mejor_y"] - particula["y"])
    componente_social_y    = c2 * r4 * (gbest_y - particula["y"])
    particula["vy"] = inercia * particula["vy"] + componente_cognitiva_y + componente_social_y

def actualizar_posicion(particula, x_min, x_max, y_min, y_max):
    particula["x"] += particula["vx"]
    particula["y"] += particula["vy"]

    if particula["x"] < x_min:
        particula["x"] = x_min
    elif particula["x"] > x_max:
        particula["x"] = x_max

    if particula["y"] < y_min:
        particula["y"] = y_min
    elif particula["y"] > y_max:
        particula["y"] = y_max

    valor_actual = funcion_objetivo(particula["x"], particula["y"])

    if valor_actual < particula["mejor_valor"]:
        particula["mejor_x"] = particula["x"]
        particula["mejor_y"] = particula["y"]
        particula["mejor_valor"] = valor_actual

def algoritmo_enjambre_particulas(
    num_particulas=30,
    iteraciones=100,
    x_min=-2.0, x_max=2.0,
    y_min=-1.0, y_max=1.0,
    inercia=1.4,
    c1=2.0,
    c2=2.0,
    reduccion_inercia=0.99,
    semilla=42
):
    random.seed(semilla)

    enjambre = []
    evaluaciones = 0

    for _ in range(num_particulas):
        p = crear_particula(x_min, x_max, y_min, y_max)
        enjambre.append(p)
        evaluaciones += 1

    mejor_global = min(enjambre, key=lambda p: p["mejor_valor"])
    gbest_x = mejor_global["mejor_x"]
    gbest_y = mejor_global["mejor_y"]
    gbest_valor = mejor_global["mejor_valor"]

    for _ in range(iteraciones):
        for particula in enjambre:
            actualizar_velocidad(particula, gbest_x, gbest_y, inercia, c1, c2)
            actualizar_posicion(particula, x_min, x_max, y_min, y_max)
            evaluaciones += 1

            if particula["mejor_valor"] < gbest_valor:
                gbest_x = particula["mejor_x"]
                gbest_y = particula["mejor_y"]
                gbest_valor = particula["mejor_valor"]

        inercia *= reduccion_inercia

    return gbest_x, gbest_y, gbest_valor, evaluaciones

def main():
    print("=" * 60)
    print("RESOLUCIÓN")
    print("=" * 60)

    inicio_estructurado = time.time()
    x_e, y_e, f_e, eval_e = busqueda_estructurada(paso=0.001)
    tiempo_estructurado = time.time() - inicio_estructurado
    
    inicio_pso = time.time()
    x_p, y_p, f_p, eval_p = algoritmo_enjambre_particulas(
        num_particulas=30,
        iteraciones=100,
        inercia=1.4,
        c1=2.0,
        c2=2.0,
        reduccion_inercia=0.99,
        semilla=42
    )
    tiempo_pso = time.time() - inicio_pso


    print("\n1) ESTRUCTURADA")
    print(f"x = {x_e:.6f}")
    print(f"y = {y_e:.6f}")
    print(f"f(x,y) = {f_e:.6f}")
    print(f"Evaluaciones = {eval_e}")
    print(f"Tiempo = {tiempo_estructurado:.4f} segundos")

    print("\n2) ENJAMBRE DE PARTÍCULAS (PSO)")
    print(f"x = {x_p:.6f}")
    print(f"y = {y_p:.6f}")
    print(f"f(x,y) = {f_p:.6f}")
    print(f"Evaluaciones = {eval_p}")
    print(f"Tiempo = {tiempo_pso:.4f} segundos")

    print("\n3) COMPARACIÓN")
    print("-" * 60)
    print(f"{'Método':<30}{'Mejor valor':<15}{'Evaluaciones':<15}{'Tiempo(s)':<10}")
    print("-" * 60)
    print(f"{'Programación estructurada':<30}{f_e:<15.6f}{eval_e:<15}{tiempo_estructurado:<10.4f}")
    print(f"{'PSO':<30}{f_p:<15.6f}{eval_p:<15}{tiempo_pso:<10.4f}")

    print("\n4) CONCLUSIÓN")
    if f_p <= f_e:
        print("El PSO resultado igual o mejor que la búsqueda estructurada,")
        print("con menos evaluaciones.")
    else:
        print("La búsqueda estructurada encontró un valor ligeramente mejor,")
        print("muchas más evaluaciones que el PSO.")

if __name__ == "__main__":
    main()

RESOLUCIÓN DEL PROBLEMA

1) ESTRUCTURADA
x = 0.090000
y = -0.713000
f(x,y) = -1.031627
Evaluaciones = 8002000
Tiempo = 4.7885 segundos

2) ENJAMBRE DE PARTÍCULAS (PSO)
x = 0.090768
y = -0.713067
f(x,y) = -1.031624
Evaluaciones = 3030
Tiempo = 0.0056 segundos

3) COMPARACIÓN
------------------------------------------------------------
Método                        Mejor valor    Evaluaciones   Tiempo(s) 
------------------------------------------------------------
Programación estructurada     -1.031627      8002000        4.7885    
PSO                           -1.031624      3030           0.0056    

4) CONCLUSIÓN
La búsqueda estructurada encontró un valor ligeramente mejor,
muchas más evaluaciones que el PSO.
